# Phase II: Original LCS Baseline



## 1. Environment and Reproducibility



## 2. Load Original Dataset




In [2]:
import pandas as pd

df = pd.read_csv("../data/telecom_customer_churn.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (7043, 38)


,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   str    
 1   Gender                             7043 non-null   str    
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   str    
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   str    
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Number of Referrals                7043 non-null   int64  
 10  Tenure in Months                   7043 non-null   int64  
 11  Offer                              3166 non-null   str    
 12  Pho

In [5]:
for col in df.columns:
    print(col)

    

Customer ID
Gender
Age
Married
Number of Dependents
City
Zip Code
Latitude
Longitude
Number of Referrals
Tenure in Months
Offer
Phone Service
Avg Monthly Long Distance Charges
Multiple Lines
Internet Service
Internet Type
Avg Monthly GB Download
Online Security
Online Backup
Device Protection Plan
Premium Tech Support
Streaming TV
Streaming Movies
Streaming Music
Unlimited Data
Contract
Paperless Billing
Payment Method
Monthly Charge
Total Charges
Total Refunds
Total Extra Data Charges
Total Long Distance Charges
Total Revenue
Customer Status
Churn Category
Churn Reason


In [6]:
for col in df.columns:
    if "churn" in col.lower():
        print(col)

Churn Category
Churn Reason


In [7]:
df["Customer Status"].value_counts(dropna=False)

Customer Status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64

## 3. Define Target and Leakage Variables



In [8]:
df["Churn Target"] = (
    df["Customer Status"] == "Churned"
).astype(int)

print("Binary target distribution:")
print(df["Churn Target"].value_counts())

print("\nBinary target percentages:")
print(
    df["Churn Target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Binary target distribution:
Churn Target
0    5174
1    1869
Name: count, dtype: int64

Binary target percentages:
Churn Target
0    73.46
1    26.54
Name: proportion, dtype: float64


In [9]:
leakage_columns = [
    "Customer Status",
    "Churn Category",
    "Churn Reason"
]

existing_leakage_columns = [
    col for col in leakage_columns
    if col in df.columns
]

print("Target-leakage columns:")
print(existing_leakage_columns)

Target-leakage columns:
['Customer Status', 'Churn Category', 'Churn Reason']


In [10]:
columns_to_exclude = [
    "Customer ID",
    "Customer Status",
    "Churn Category",
    "Churn Reason",
    "Churn Target"
]

columns_to_exclude = [
    col for col in columns_to_exclude
    if col in df.columns
]

X_raw = df.drop(columns=columns_to_exclude)
y = df["Churn Target"]

print("Raw feature shape:", X_raw.shape)
print("Target shape:", y.shape)
print("Excluded columns:", columns_to_exclude)

Raw feature shape: (7043, 34)
Target shape: (7043,)
Excluded columns: ['Customer ID', 'Customer Status', 'Churn Category', 'Churn Reason', 'Churn Target']


### Target Definition

Customer Status contains three categories: Stayed, Churned, and Joined. A binary target was created for the churn-classification task. Churned customers were assigned a value of 1, while Stayed and Joined customers were assigned a value of 0.

Customer Status, Churn Category, and Churn Reason were excluded from the predictor variables because they contain direct or post-outcome information about customer churn and could introduce target leakage. Customer ID was also excluded because it is an identifier rather than a predictive feature.

## 4. Minimal LCS Compatibility Processing



In [13]:
print(df.dtypes)




Customer ID                              str
Gender                                   str
Age                                    int64
Married                                  str
Number of Dependents                   int64
City                                     str
Zip Code                               int64
Latitude                             float64
Longitude                            float64
Number of Referrals                    int64
Tenure in Months                       int64
Offer                                    str
Phone Service                            str
Avg Monthly Long Distance Charges    float64
Multiple Lines                           str
Internet Service                         str
Internet Type                            str
Avg Monthly GB Download              float64
Online Security                          str
Online Backup                            str
Device Protection Plan                   str
Premium Tech Support                     str
Streaming 

In [16]:
numeric_columns = (
    X_raw.select_dtypes(
        exclude=["object"]
    )
    .columns
    .tolist()
)

print("Numeric Columns:")
print(numeric_columns)

print("\nNumber of numeric columns:")
print(len(numeric_columns))

Numeric Columns:
['Age', 'Number of Dependents', 'Zip Code', 'Latitude', 'Longitude', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue']

Number of numeric columns:
15


In [17]:
categorical_columns = (
    X_raw.select_dtypes(include=["object"])
    .columns
    .tolist()
)

print("Categorical Columns:")
print(categorical_columns)

print("\nNumber of categorical columns:")
print(len(categorical_columns))


Categorical Columns:
['Gender', 'Married', 'City', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method']

Number of categorical columns:
19


C:\Users\ACER\AppData\Local\Temp\ipykernel_7008\3951263282.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_raw.select_dtypes(include=["object"])


In [18]:
X_raw.shape


(7043, 34)

In [19]:
categorical_columns = (
    X_train_raw
    .select_dtypes(include=["string"])
    .columns
    .tolist()
)

numeric_columns = [
    col for col in X_train_raw.columns
    if col not in categorical_columns
]

print("Categorical columns:", len(categorical_columns))
print("Numeric columns:", len(numeric_columns))
print("Total columns:", len(X_train_raw.columns))

Categorical columns: 19
Numeric columns: 15
Total columns: 34


In [20]:
from sklearn.preprocessing import OrdinalEncoder

In [21]:
X_train_lcs = X_train_raw.copy()
X_test_lcs = X_test_raw.copy()

In [22]:
baseline_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-2
)

X_train_lcs[categorical_columns] = (
    baseline_encoder.fit_transform(
        X_train_raw[categorical_columns]
    )
)

X_test_lcs[categorical_columns] = (
    baseline_encoder.transform(
        X_test_raw[categorical_columns]
    )
)

print("Categorical variables encoded successfully.")

Categorical variables encoded successfully.


In [23]:
X_train_lcs = X_train_lcs.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test_lcs = X_test_lcs.apply(
    pd.to_numeric,
    errors="coerce"
)

print("Training data types:")
print(X_train_lcs.dtypes.value_counts())

print("\nTesting data types:")
print(X_test_lcs.dtypes.value_counts())


Training data types:
float64    28
int64       6
Name: count, dtype: int64

Testing data types:
float64    28
int64       6
Name: count, dtype: int64


In [24]:
remaining_text_train = (
    X_train_lcs
    .select_dtypes(include=["string", "object"])
    .columns
    .tolist()
)

remaining_text_test = (
    X_test_lcs
    .select_dtypes(include=["string", "object"])
    .columns
    .tolist()
)

print(
    "Text columns remaining in training data:",
    remaining_text_train
)

print(
    "Text columns remaining in testing data:",
    remaining_text_test
)

print(
    "Missing feature values in training data:",
    int(X_train_lcs.isna().sum().sum())
)

print(
    "Missing feature values in testing data:",
    int(X_test_lcs.isna().sum().sum())
)

Text columns remaining in training data: []
Text columns remaining in testing data: []
Missing feature values in training data: 1736
Missing feature values in testing data: 472


In [25]:
X_train_lcs_array = X_train_lcs.to_numpy(
    dtype=float
)

X_test_lcs_array = X_test_lcs.to_numpy(
    dtype=float
)

y_train_lcs_array = y_train.to_numpy(
    dtype=int
)

y_test_lcs_array = y_test.to_numpy(
    dtype=int
)

feature_headers = X_train_lcs.columns.to_numpy()

print(
    "Training feature array:",
    X_train_lcs_array.shape
)

print(
    "Testing feature array:",
    X_test_lcs_array.shape
)

print(
    "Training target array:",
    y_train_lcs_array.shape
)

print(
    "Testing target array:",
    y_test_lcs_array.shape
)

Training feature array: (5634, 34)
Testing feature array: (1409, 34)
Training target array: (5634,)
Testing target array: (1409,)


### Minimal eLCS Compatibility Processing

The original eLCS implementation requires a fully numeric feature array and a one-dimensional numeric target array. The baseline dataset contained 19 categorical predictors and 15 numerical predictors. The categorical predictors were converted to numeric codes using an ordinal encoder, while the numerical predictors were retained without scaling or feature engineering.

The encoder was fitted only on the training data and then applied to the testing data to avoid data leakage. Previously unseen testing categories are represented by -1, and missing categorical values are represented by -2. The resulting feature and target datasets were converted to NumPy arrays as required by the original eLCS implementation.

## 5. Stratified Train-Test Split



In [11]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train_raw.shape)
print("Testing features:", X_test_raw.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training features: (5634, 34)
Testing features: (1409, 34)

Training target distribution:
Churn Target
0    4139
1    1495
Name: count, dtype: int64

Testing target distribution:
Churn Target
0    1035
1     374
Name: count, dtype: int64


In [12]:
print("Training percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting percentages:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training percentages:
Churn Target
0    73.46
1    26.54
Name: proportion, dtype: float64

Testing percentages:
Churn Target
0    73.46
1    26.54
Name: proportion, dtype: float64


### Train-Test Strategy

An 80/20 train-test split was used for the baseline experiments. Stratification was applied to preserve the class distribution of churned and non-churned customers in both training and testing datasets. A fixed random seed (42) was used to ensure reproducibility.

## 6. Original LCS Configuration



In [26]:
from skeLCS import eLCS

print("Original eLCS imported successfully")

Original eLCS imported successfully


## 7. Baseline Training



In [27]:
baseline_lcs = eLCS()
 
print(baseline_lcs.get_params())

{'N': 1000, 'acc_sub': 0.99, 'beta': 0.2, 'chi': 0.8, 'delta': 0.1, 'discrete_attribute_limit': 10, 'do_GA_subsumption': True, 'do_correct_set_subsumption': False, 'fitness_reduction': 0.1, 'init_fit': 0.01, 'learning_iterations': 10000, 'match_for_missingness': False, 'mu': 0.04, 'nu': 5, 'p_spec': 0.5, 'random_state': None, 'reboot_filename': None, 'selection_method': 'tournament', 'specified_attributes': array([], dtype=float64), 'theta_GA': 25, 'theta_del': 20, 'theta_sel': 0.5, 'theta_sub': 20, 'track_accuracy_while_fit': False}


In [28]:
import time

training_start_time = time.time()

baseline_lcs.fit(
    X_train_lcs_array,
    y_train_lcs_array
)

training_end_time = time.time()

baseline_training_seconds = (
    training_end_time - training_start_time
)

print("Original eLCS baseline training completed.")
print(
    "Training time in seconds:",
    round(baseline_training_seconds, 2)
)

Original eLCS baseline training completed.
Training time in seconds: 28.81


## 8. Baseline Evaluation



In [29]:
baseline_predictions = baseline_lcs.predict(
    X_test_lcs_array
)

print("Baseline predictions generated.")
print("Number of predictions:", len(baseline_predictions))
print("First 10 predictions:", baseline_predictions[:10])





Baseline predictions generated.
Number of predictions: 1409
First 10 predictions: [0 0 0 0 0 0 0 0 0 1]


In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

baseline_accuracy = accuracy_score(
    y_test_lcs_array,
    baseline_predictions
)

baseline_precision = precision_score(
    y_test_lcs_array,
    baseline_predictions,
    zero_division=0
)

baseline_recall = recall_score(
    y_test_lcs_array,
    baseline_predictions,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test_lcs_array,
    baseline_predictions,
    zero_division=0
)

baseline_balanced_accuracy = balanced_accuracy_score(
    y_test_lcs_array,
    baseline_predictions
)

baseline_confusion_matrix = confusion_matrix(
    y_test_lcs_array,
    baseline_predictions
)

print("Original eLCS Baseline Results")
print("--------------------------------")
print("Accuracy:", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall:", round(baseline_recall, 4))
print("F1-score:", round(baseline_f1, 4))
print(
    "Balanced accuracy:",
    round(baseline_balanced_accuracy, 4)
)

print("\nConfusion matrix:")
print(baseline_confusion_matrix)

print("\nClassification report:")
print(
    classification_report(
        y_test_lcs_array,
        baseline_predictions,
        zero_division=0
    )
)

Original eLCS Baseline Results
--------------------------------
Accuracy: 0.807
Precision: 0.6962
Recall: 0.484
F1-score: 0.571
Balanced accuracy: 0.7038

Confusion matrix:
[[956  79]
 [193 181]]

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.92      0.88      1035
           1       0.70      0.48      0.57       374

    accuracy                           0.81      1409
   macro avg       0.76      0.70      0.72      1409
weighted avg       0.80      0.81      0.79      1409



In [1]:
import sys
import platform
import pandas as pd
import numpy as np
import sklearn
import scipy

print("Python:", sys.version)
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)

Python: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.22631-SP0
pandas: 3.0.5
numpy: 2.5.2
scikit-learn: 1.9.1
scipy: 1.18.1


In [31]:
baseline_probabilities = baseline_lcs.predict_proba(
    X_test_lcs_array
)

print("Probability output shape:")
print(baseline_probabilities.shape)

print("\nFirst five probability outputs:")
print(baseline_probabilities[:5])



Probability output shape:
(1409, 2)

First five probability outputs:
[[9.92747663e-01 7.25233686e-03]
 [9.32430233e-01 6.75697667e-02]
 [9.41455955e-01 5.85440451e-02]
 [9.99965196e-01 3.48044111e-05]
 [1.00000000e+00 0.00000000e+00]]


In [32]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

positive_class_probabilities = baseline_probabilities[:, 1]

baseline_roc_auc = roc_auc_score(
    y_test_lcs_array,
    positive_class_probabilities
)

baseline_pr_auc = average_precision_score(
    y_test_lcs_array,
    positive_class_probabilities
)

print("ROC-AUC:", round(baseline_roc_auc, 4))
print("PR-AUC:", round(baseline_pr_auc, 4))


ROC-AUC: 0.8277
PR-AUC: 0.6579


## 9. Export Results


In [33]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

baseline_results_directory = Path("../results/baseline")
baseline_results_directory.mkdir(parents=True, exist_ok=True)

print("Results folder ready:")
print(baseline_results_directory.resolve())

Results folder ready:
C:\Users\ACER\Documents\ENGE707-2026-S2-Low-Battery\results\baseline


In [34]:
baseline_metrics = {
    "model": "Original eLCS",
    "accuracy": baseline_accuracy,
    "precision": baseline_precision,
    "recall": baseline_recall,
    "f1_score": baseline_f1,
    "balanced_accuracy": baseline_balanced_accuracy,
    "training_time_seconds": baseline_training_seconds
}

if "baseline_roc_auc" in globals():
    baseline_metrics["roc_auc"] = baseline_roc_auc

if "baseline_pr_auc" in globals():
    baseline_metrics["pr_auc"] = baseline_pr_auc

baseline_metrics_df = pd.DataFrame(
    [baseline_metrics]
)

baseline_metrics_df.to_csv(
    baseline_results_directory / "baseline_metrics.csv",
    index=False
)

baseline_metrics_df

,model,accuracy,precision,recall,f1_score,balanced_accuracy,training_time_seconds,roc_auc,pr_auc
0,Original eLCS,0.806955,0.696154,0.483957,0.570978,0.703814,28.810128,0.827712,0.657853


In [35]:
baseline_predictions_df = pd.DataFrame({
    "actual_target": y_test_lcs_array,
    "predicted_target": baseline_predictions
})

if "baseline_probabilities" in globals():
    baseline_predictions_df["probability_non_churn"] = (
        baseline_probabilities[:, 0]
    )
    
    baseline_predictions_df["probability_churn"] = (
        baseline_probabilities[:, 1]
    )

baseline_predictions_df.to_csv(
    baseline_results_directory / "baseline_predictions.csv",
    index=False
)

print("Baseline predictions saved.")
baseline_predictions_df.head()

Baseline predictions saved.


,actual_target,predicted_target,probability_non_churn,probability_churn
0,0,0,0.992748,0.007252
1,0,0,0.932430,0.067570
2,0,0,0.941456,0.058544
3,0,0,0.999965,0.000035
4,0,0,1.000000,0.000000


In [36]:
baseline_confusion_matrix_df = pd.DataFrame(
    baseline_confusion_matrix,
    index=["Actual Non-Churn", "Actual Churn"],
    columns=["Predicted Non-Churn", "Predicted Churn"]
)

baseline_confusion_matrix_df.to_csv(
    baseline_results_directory / "baseline_confusion_matrix.csv"
)

baseline_confusion_matrix_df


,Predicted Non-Churn,Predicted Churn
Actual Non-Churn,956,79
Actual Churn,193,181


In [37]:
baseline_parameters = baseline_lcs.get_params()

serialisable_parameters = {}

for parameter, value in baseline_parameters.items():
    if isinstance(value, np.ndarray):
        serialisable_parameters[parameter] = value.tolist()
    elif isinstance(value, np.generic):
        serialisable_parameters[parameter] = value.item()
    else:
        serialisable_parameters[parameter] = value

with open(
    baseline_results_directory / "baseline_parameters.json",
    "w",
    encoding="utf-8"
) as parameter_file:
    json.dump(
        serialisable_parameters,
        parameter_file,
        indent=4
    )

print("Original eLCS parameters saved.")

Original eLCS parameters saved.


In [38]:
print("Exported baseline files:")

for file_path in sorted(baseline_results_directory.iterdir()):
    print(file_path.name)

Exported baseline files:
baseline_confusion_matrix.csv
baseline_metrics.csv
baseline_parameters.json
baseline_predictions.csv
